# GraphRAG Text-to-SQL Pipeline

A modular pipeline for the Spider Text-to-SQL benchmark, combining **Graph-based schema retrieval (GraphRAG)** with a **quantized LLM** (Qwen2.5-Coder-7B-Instruct) for SQL generation.

**Architecture:**
```
Question
   │
   ▼
Semantic Schema Linking  ← BGE-M3 embeddings, n-gram matching
   │
   ▼
Graph Path Tracing       ← NetworkX shortest-path on schema graph
   │
   ▼
Schema Context Builder   ← CREATE TABLE DDL with FK annotations
   │
   ▼
Prompt Builder           ← Structured prompt with few-shot examples
   │
   ▼
LLM (Qwen2.5-Coder 7B)  ← 4-bit quantized, greedy decode
   │
   ▼
SQL Cleaner              ← Strip dialect quirks, fix aliases
   │
   ▼
Official Spider Eval     ← EM + EX via evaluation.py
```

**Results:** Exact Match (EM): `0.593` | Execution Accuracy (EX): `0.622`


## 0. Install Dependencies

In [ ]:
%pip install -U bitsandbytes>=0.46.1 sentence-transformers transformers networkx pandas torch tqdm

# Download Spider official evaluation scripts
!wget -nc https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
!wget -nc https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py

## 1. Imports & Logging

In [ ]:
import argparse
import csv
import json
import logging
import random
import re
import subprocess
import sys
from dataclasses import dataclass, field
from itertools import product
from pathlib import Path
from typing import Optional
import pandas as pd

import networkx as nx
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

## 2. Configuration (`config.py`)

In [ ]:
@dataclass
class PipelineConfig:
    # --- Paths ---
    data_path: Path = Path("/kaggle/input/datasets/alrette/spiderdataset/spider_data")
    predictions_file: Path = Path("predictions.txt")

    @property
    def tables_json(self) -> Path:
        return self.data_path / "tables.json"

    @property
    def dev_json(self) -> Path:
        return self.data_path / "dev.json"

    @property
    def db_dir(self) -> Path:
        return self.data_path / "database"

    @property
    def gold_sql(self) -> Path:
        p = self.data_path / "dev_gold.sql"
        return p if p.exists() else self.data_path / "dev_gold"

    @property
    def train_json(self) -> Path:
        return self.data_path / "train_spider.json"

    # --- Models ---
    embedding_model: str = "BAAI/bge-m3"
    llm_model: str = "Qwen/Qwen2.5-Coder-7B-Instruct"

    # --- Quantization ---
    load_in_4bit: bool = True
    bnb_double_quant: bool = True
    bnb_quant_type: str = "nf4"
    bnb_compute_dtype: torch.dtype = torch.bfloat16

    # --- Schema Linking ---
    semantic_similarity_threshold: float = 0.35
    max_ngram: int = 3
    top_k_columns: int = 5
    top_k_tables: int = 3

    # --- Few-shot ---
    few_shot_k: int = 3
    few_shot_same_db_first: bool = True

    # --- Generation ---
    max_new_tokens: int = 200
    temperature: float = 0.0

    # --- Mode ---
    use_full_schema_bypass: bool = False

    # --- Reproducibility ---
    seed: int = 42

## 3. Schema Processing (`schema.py`)

Load Spider's `tables.json` and build a NetworkX column-level graph.

In [ ]:
def load_spider_schema(json_path: Path) -> "pd.DataFrame":
    import pandas as pd
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = []
    for db in data:
        db_id = db["db_id"]
        table_names = db["table_names_original"]
        column_names = db["column_names_original"]
        column_types = db["column_types"]
        primary_keys = set(db["primary_keys"])
        fk_map = {fk[0]: fk[1] for fk in db["foreign_keys"]}

        for col_idx, (table_idx, col_name) in enumerate(column_names):
            table_name = table_names[table_idx] if table_idx != -1 else "ALL"
            fk_relation = "-"
            if col_idx in fk_map:
                target_idx = fk_map[col_idx]
                t_table_idx, t_col_name = column_names[target_idx]
                fk_relation = f"{table_names[t_table_idx]}.{t_col_name}"
            records.append({
                "Database": db_id, "Table": table_name, "Column": col_name,
                "Type": column_types[col_idx],
                "PK": "Yes" if col_idx in primary_keys else "No",
                "FK_Relation": fk_relation,
            })
    return pd.DataFrame(records)


def build_schema_graph(df) -> nx.Graph:
    """Build a column-level graph with belongs_to_pk and foreign_key edges."""
    G = nx.Graph()
    pk_map: dict[str, dict[str, str]] = {}
    df_clean = df[df["Column"] != "*"].copy()

    for _, row in df_clean.iterrows():
        db, table, col = str(row["Database"]).strip(), str(row["Table"]).strip(), str(row["Column"]).strip()
        node_id = f"{db}.{table}.{col}"
        is_pk = row["PK"] == "Yes"
        G.add_node(node_id, type="column", database=db, table=table, column=col, column_type=str(row["Type"]).strip(), is_pk=is_pk)
        if db not in pk_map:
            pk_map[db] = {}
        if is_pk:
            pk_map[db][table] = node_id

    for _, row in df_clean.iterrows():
        db, table, col = str(row["Database"]).strip(), str(row["Table"]).strip(), str(row["Column"]).strip()
        node_id = f"{db}.{table}.{col}"
        fk_rel = str(row["FK_Relation"]).strip()
        if table in pk_map.get(db, {}) and node_id != pk_map[db][table]:
            G.add_edge(node_id, pk_map[db][table], relation="belongs_to_pk")
        if fk_rel != "-":
            target_node_id = f"{db}.{fk_rel}"
            if target_node_id in G:
                G.add_edge(node_id, target_node_id, relation="foreign_key")

    logger.info("Column graph built: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges())
    return G

## 4. GraphRAG Schema Linking (`retrieval.py`)

Two-stage retrieval: table-level → column-level, with graph path tracing and pruning.

In [ ]:
# ---------------------------------------------------------------------------
# Schema Index
# ---------------------------------------------------------------------------

@dataclass
class SchemaIndex:
    db_name: str
    tables: list[str]
    table_embeddings: Optional[torch.Tensor]
    columns: list[str]
    col_node_ids: list[str]
    col_table_map: list[str]
    col_embeddings: Optional[torch.Tensor]


def build_schema_index(graph: nx.Graph, db_name: str, embed_model: SentenceTransformer) -> SchemaIndex:
    col_nodes = [(n, d) for n, d in graph.nodes(data=True)
                 if d.get("database") == db_name and d.get("type") == "column"]
    if not col_nodes:
        return SchemaIndex(db_name, [], None, [], [], [], None)

    col_node_ids = [n for n, _ in col_nodes]
    columns = [d["column"] for _, d in col_nodes]
    col_table_map = [d["table"] for _, d in col_nodes]
    col_texts = [f"{t}.{c}" for t, c in zip(col_table_map, columns)]
    col_embeddings = embed_model.encode(col_texts, convert_to_tensor=True)

    tables = list(dict.fromkeys(col_table_map))
    table_embeddings = embed_model.encode(tables, convert_to_tensor=True)

    return SchemaIndex(db_name=db_name, tables=tables, table_embeddings=table_embeddings,
                       columns=columns, col_node_ids=col_node_ids, col_table_map=col_table_map,
                       col_embeddings=col_embeddings)


def precompute_column_embeddings(graph, db_name, embed_model):
    idx = build_schema_index(graph, db_name, embed_model)
    return idx.columns, idx.col_embeddings


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _extract_phrases(query: str, max_ngram: int) -> list[str]:
    clean = re.sub(r"[^\w\s]", "", query.lower())
    words = clean.split()
    return list({" ".join(words[i: i + length])
                 for length in range(1, max_ngram + 1)
                 for i in range(len(words) - length + 1)})


def _top_k_indices(score_row: torch.Tensor, k: int) -> list[int]:
    k = min(k, score_row.size(0))
    return torch.topk(score_row, k).indices.tolist()


# ---------------------------------------------------------------------------
# Stage 1: Table retrieval
# ---------------------------------------------------------------------------

def retrieve_candidate_tables(query: str, index: SchemaIndex,
                               embed_model: SentenceTransformer, cfg: PipelineConfig) -> list[str]:
    if cfg.top_k_tables == 0 or index.table_embeddings is None:
        return index.tables
    phrases = _extract_phrases(query, cfg.max_ngram)
    if not phrases:
        return index.tables
    phrase_emb = embed_model.encode(phrases, convert_to_tensor=True)
    scores = util.cos_sim(phrase_emb, index.table_embeddings)
    max_scores, _ = scores.max(dim=0)
    return [index.tables[i] for i in _top_k_indices(max_scores, cfg.top_k_tables)]


# ---------------------------------------------------------------------------
# Stage 2: Column retrieval
# ---------------------------------------------------------------------------

def retrieve_candidate_columns(query: str, index: SchemaIndex, candidate_tables: list[str],
                                embed_model: SentenceTransformer, cfg: PipelineConfig) -> list[str]:
    if index.col_embeddings is None:
        return []
    candidate_table_set = set(candidate_tables)
    col_mask = torch.tensor([t in candidate_table_set for t in index.col_table_map], dtype=torch.bool)
    if not col_mask.any():
        return []
    filtered_emb = index.col_embeddings[col_mask]
    filtered_cols = [c for c, m in zip(index.columns, col_mask.tolist()) if m]
    phrases = _extract_phrases(query, cfg.max_ngram)
    if not phrases:
        return []
    phrase_emb = embed_model.encode(phrases, convert_to_tensor=True)
    scores = util.cos_sim(phrase_emb, filtered_emb)
    detected: set[str] = set()
    if cfg.top_k_columns > 0:
        for i in range(len(phrases)):
            for col_idx in _top_k_indices(scores[i], cfg.top_k_columns):
                detected.add(filtered_cols[col_idx])
    else:
        for i in range(len(phrases)):
            best_score = torch.max(scores[i]).item()
            if best_score > cfg.semantic_similarity_threshold:
                detected.add(filtered_cols[torch.argmax(scores[i]).item()])
    return list(detected)


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------

def semantic_schema_linking(graph, db_name, user_query, all_columns, col_embeddings,
                             embed_model, cfg, index=None):
    if index is None:
        col_nodes = [(n, d) for n, d in graph.nodes(data=True)
                     if d.get("database") == db_name and d.get("type") == "column"]
        tables = list(dict.fromkeys(d["table"] for _, d in col_nodes))
        col_table_map = [d["table"] for _, d in col_nodes]
        table_embeddings = embed_model.encode(tables, convert_to_tensor=True) if tables else None
        index = SchemaIndex(db_name=db_name, tables=tables, table_embeddings=table_embeddings,
                            columns=all_columns, col_node_ids=[n for n, _ in col_nodes],
                            col_table_map=col_table_map, col_embeddings=col_embeddings)
    candidate_tables = retrieve_candidate_tables(user_query, index, embed_model, cfg)
    return retrieve_candidate_columns(user_query, index, candidate_tables, embed_model, cfg)


def trace_schema_paths(graph, db_name, detected_columns):
    column_nodes = [n for n, d in graph.nodes(data=True)
                    if d.get("database") == db_name and d.get("column") in detected_columns]
    paths, relations = [], []
    for i in range(len(column_nodes)):
        for j in range(i + 1, len(column_nodes)):
            try:
                path = nx.shortest_path(graph, column_nodes[i], column_nodes[j])
                paths.append(path)
                for k in range(len(path) - 1):
                    edge_data = graph.get_edge_data(path[k], path[k + 1])
                    relations.append({"from": path[k], "to": path[k + 1],
                                      "relation": edge_data.get("relation") if edge_data else None})
            except nx.NetworkXNoPath:
                continue
    return column_nodes, paths, relations


def prune_path_nodes(graph, detected_column_names, all_path_nodes):
    kept = []
    for node in all_path_nodes:
        d = graph.nodes.get(node, {})
        if not d:
            continue
        if d.get("column", "").lower() in detected_column_names:
            kept.append(node); continue
        if d.get("is_pk", False):
            kept.append(node); continue
        has_fk = any(graph.get_edge_data(node, nb, {}).get("relation") == "foreign_key"
                     for nb in graph.neighbors(node))
        if has_fk:
            kept.append(node)
    return kept


def build_schema_context(graph, column_nodes):
    tables: dict[str, list[dict]] = {}
    for node in column_nodes:
        d = graph.nodes[node]
        tables.setdefault(d["table"], []).append(
            {"col": d["column"], "col_type": d["column_type"], "is_pk": d["is_pk"], "node_id": node})
    lines = []
    for table, cols in tables.items():
        lines.append(f"CREATE TABLE {table} (")
        col_lines, fk_lines = [], []
        for col_data in cols:
            col, col_type, is_pk = col_data["col"], col_data["col_type"], col_data["is_pk"]
            col_lines.append(f"    {col} {col_type} PRIMARY KEY" if is_pk else f"    {col} {col_type}")
            for neighbor in graph.neighbors(col_data["node_id"]):
                edge = graph.get_edge_data(col_data["node_id"], neighbor)
                if edge and edge.get("relation") == "foreign_key":
                    n_data = graph.nodes[neighbor]
                    if n_data["is_pk"] and n_data["table"] != table:
                        fk_lines.append(f"    FOREIGN KEY ({col}) REFERENCES {n_data['table']}({n_data['column']})")
        lines.append(",\n".join(col_lines + fk_lines))
        lines.append(");\n")
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Evaluation helpers
# ---------------------------------------------------------------------------

_SQL_STOPWORDS = frozenset({
    "select", "from", "where", "join", "on", "as", "and", "or", "not",
    "in", "is", "null", "like", "between", "exists", "case", "when",
    "then", "else", "end", "order", "by", "group", "having", "limit",
    "offset", "union", "intersect", "except", "distinct", "all", "asc",
    "desc", "count", "sum", "avg", "min", "max", "inner", "left", "right",
    "outer", "full", "cross", "natural", "using", "with", "create", "table",
    "insert", "update", "delete", "set", "values", "primary", "key",
    "foreign", "references", "index", "view", "drop", "alter", "add",
    "t1", "t2", "t3", "t4", "t5",
})


def _parse_gold_elements(gold_sql, graph, db_name):
    known_tables, known_columns = set(), set()
    for _, d in graph.nodes(data=True):
        if d.get("database") != db_name:
            continue
        known_tables.add(d["table"].lower())
        col = d["column"].lower()
        if col != "*":
            known_columns.add(col)
    raw_tokens = re.split(r"[\s\(\),;=<>!\"']+", gold_sql.lower())
    tokens = {t.lstrip(".") for t in raw_tokens if t and t not in _SQL_STOPWORDS}
    gold_elements = set()
    for token in tokens:
        if token in known_tables:
            gold_elements.add(token)
        if token in known_columns:
            gold_elements.add(token)
    return gold_elements


def evaluate_schema_linking(gold_sql, pruned_nodes, graph, db_name):
    gold_elements = _parse_gold_elements(gold_sql, graph, db_name)
    if not gold_elements:
        return 1.0, 1.0
    retrieved = set()
    for node in pruned_nodes:
        d = graph.nodes[node]
        retrieved.add(d["table"].lower())
        col = d["column"].lower()
        if col != "*":
            retrieved.add(col)
    hits = gold_elements & retrieved
    recall = len(hits) / len(gold_elements)
    precision = len(hits) / len(retrieved) if retrieved else 0.0
    return recall, precision

## 5. Prompt & SQL Generation (`generation.py`)

In [ ]:
# Compiled regex patterns
_COMPLEX_KEYWORDS = frozenset(["JOIN", "INTERSECT", "EXCEPT", "UNION"])
_HAS_SUBQUERY = re.compile(r"\bSELECT\b.*\bSELECT\b", re.IGNORECASE | re.DOTALL)
_NULLS_PATTERN = re.compile(r"\s+NULLS\s+(LAST|FIRST)", re.IGNORECASE)
_ILIKE_PATTERN = re.compile(r"\bILIKE\b", re.IGNORECASE)
_CAST_PATTERN = re.compile(r"::text\b", re.IGNORECASE)
_ALIAS_INLINE = re.compile(r"(?i)\s+AS\s+[a-zA-Z0-9_]+(?=[\s,]|FROM|$)")
_TABLE_PREFIX = re.compile(r"\b[a-zA-Z_][a-zA-Z0-9_]*\.(?=[a-zA-Z_])")
_WHITESPACE = re.compile(r"\s+")


def load_model_and_tokenizer(cfg: PipelineConfig):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=cfg.load_in_4bit,
        bnb_4bit_use_double_quant=cfg.bnb_double_quant,
        bnb_4bit_quant_type=cfg.bnb_quant_type,
        bnb_4bit_compute_dtype=cfg.bnb_compute_dtype,
    )
    tokenizer = AutoTokenizer.from_pretrained(cfg.llm_model)
    model = AutoModelForCausalLM.from_pretrained(
        cfg.llm_model, quantization_config=bnb_config, device_map="auto", trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    torch.cuda.empty_cache()
    return model, tokenizer


def build_prompt(question, schema_context, extracted_values, few_shot_block=""):
    return (
        "### Task\n"
        "Generate a valid SQLite query to answer the following question.\n"
        "### Database Schema\n"
        f"{schema_context}\n"
        f"{few_shot_block}"
        "### Question\n"
        f"{question}\n"
        "### Instructions\n"
        "1. Only use the tables and columns provided in the schema above. DO NOT make up names.\n"
        "2. JOIN ALIASING (CRITICAL): If you use JOINs, you MUST explicitly declare table aliases "
        "in the FROM clause using `AS T1`, `AS T2`, etc.\n"
        "3. SINGLE TABLE RULES: If querying a single table (NO JOIN), DO NOT use any table aliases.\n"
        "4. ALWAYS use `COUNT(*)` instead of `COUNT(column)`.\n"
        "5. SET OPERATIONS: Use `INTERSECT` to find items matching two separate criteria. "
        "Use `EXCEPT` for exclusionary conditions.\n"
        "6. Output ONLY the raw SQL query. Do not add markdown or explanations.\n"
        "### Answer\n"
        "```sql\n"
    )


def _clean_sql(sql: str) -> str:
    sql = sql.replace("```sql", "").replace("```", "").replace("`", "").strip()
    match = re.search(r"(SELECT\s+.*?)(?:;|$)", sql, re.DOTALL | re.IGNORECASE)
    sql = match.group(1).strip() if match else sql.strip()
    if not sql:
        return "SELECT 1"
    sql = _NULLS_PATTERN.sub("", sql)
    sql = _ILIKE_PATTERN.sub("LIKE", sql)
    sql = _CAST_PATTERN.sub("", sql)
    is_simple = (not any(kw in sql.upper() for kw in _COMPLEX_KEYWORDS)
                 and not _HAS_SUBQUERY.search(sql))
    if is_simple:
        sql = _ALIAS_INLINE.sub("", sql)
        parts = re.split(r"\bFROM\b", sql, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            select_part, from_part = parts[0], parts[1].strip()
            select_part = _TABLE_PREFIX.sub("", select_part)
            table_match = re.match(r"^([a-zA-Z0-9_]+)", from_part)
            if table_match:
                table_name = table_match.group(1)
                kw_match = re.search(r"(?i)\b(WHERE|GROUP|ORDER|HAVING|LIMIT)\b.*", from_part, re.DOTALL)
                from_part = f"{table_name} {kw_match.group(0)}" if kw_match else table_name
            sql = f"{select_part} FROM {from_part}"
    return _WHITESPACE.sub(" ", sql).strip()


def generate_sql(prompt, model, tokenizer, cfg):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=cfg.max_new_tokens, temperature=cfg.temperature,
            do_sample=False, pad_token_id=tokenizer.eos_token_id)
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return _clean_sql(generated_text)

## 6. Few-Shot Retrieval (`few_shot.py`)

In [ ]:
@dataclass
class FewShotExample:
    question: str
    sql: str
    db_id: str


@dataclass
class FewShotIndex:
    examples: list[FewShotExample]
    embeddings: torch.Tensor


def build_few_shot_index(train_json_path: Path, embed_model: SentenceTransformer) -> FewShotIndex:
    with open(train_json_path, "r", encoding="utf-8") as f:
        train_data = json.load(f)
    examples = [FewShotExample(question=item["question"], sql=item["query"], db_id=item["db_id"])
                for item in train_data]
    logger.info("Encoding %d training questions for few-shot index …", len(examples))
    embeddings = embed_model.encode([e.question for e in examples], convert_to_tensor=True,
                                     show_progress_bar=True, batch_size=256)
    logger.info("Few-shot index ready.")
    return FewShotIndex(examples=examples, embeddings=embeddings)


def retrieve_few_shot_examples(question, db_id, index, embed_model, cfg):
    if cfg.few_shot_k == 0 or index is None:
        return []
    query_emb = embed_model.encode(question, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, index.embeddings)[0]
    pool_size = min(cfg.few_shot_k * 4 if cfg.few_shot_same_db_first else cfg.few_shot_k, len(index.examples))
    top_indices = torch.topk(scores, pool_size).indices.tolist()
    candidates = [index.examples[i] for i in top_indices]
    if cfg.few_shot_same_db_first:
        same_db = [e for e in candidates if e.db_id == db_id]
        other_db = [e for e in candidates if e.db_id != db_id]
        candidates = same_db + other_db
    return candidates[:cfg.few_shot_k]


def format_few_shot_block(examples):
    if not examples:
        return ""
    lines = ["### Examples\n"]
    for i, ex in enumerate(examples, start=1):
        lines.append(f"-- Example {i}")
        lines.append(f"-- Q: {ex.question}")
        lines.append("-- A:")
        lines.append(ex.sql.strip())
        lines.append("")
    return "\n".join(lines) + "\n" 

## 7. Precision Sweep (`sweep.py`)

Optimise `top_k_tables` × `top_k_columns` using embedding-only retrieval (no LLM needed).

In [ ]:
# Grid to sweep
TOP_K_TABLES_VALUES  = [3, 4, 5]
TOP_K_COLUMNS_VALUES = [2, 3, 5, 7, 10]
RECALL_THRESHOLD = 0.90


@dataclass
class SweepResult:
    top_k_tables: int
    top_k_columns: int
    avg_recall: float
    avg_precision: float
    f1: float
    recall_weighted_f1: float
    meets_recall_target: bool
    n_samples: int


def _run_sweep_combination(top_k_tables, top_k_columns, dev_subset, graph, schema_cache,
                            embed_model, base_cfg):
    cfg = PipelineConfig(data_path=base_cfg.data_path, top_k_tables=top_k_tables,
                         top_k_columns=top_k_columns, few_shot_k=0)
    recalls, precisions = [], []
    for item in dev_subset:
        db_id, question, gold_sql = item["db_id"], item["question"], item["query"]
        index = schema_cache[db_id]
        detected_cols = semantic_schema_linking(
            graph, db_id, question, all_columns=index.columns, col_embeddings=index.col_embeddings,
            embed_model=embed_model, cfg=cfg, index=index)
        if not detected_cols:
            column_nodes = [n for n, d in graph.nodes(data=True)
                            if d.get("database") == db_id and d.get("type") == "column"]
        else:
            c_nodes, paths, _ = trace_schema_paths(graph, db_id, detected_cols)
            all_path_nodes = list(set(c_nodes + [node for path in paths for node in path]))
            detected_col_names = {c.lower() for c in detected_cols}
            column_nodes = prune_path_nodes(graph, detected_col_names, all_path_nodes) or all_path_nodes
        if not column_nodes:
            recalls.append(0.0); precisions.append(0.0); continue
        r, p = evaluate_schema_linking(gold_sql, column_nodes, graph, db_id)
        recalls.append(r); precisions.append(p)
    avg_r, avg_p = float(np.mean(recalls)), float(np.mean(precisions))
    f1 = (2 * avg_r * avg_p / (avg_r + avg_p)) if (avg_r + avg_p) > 0 else 0.0
    meets_target = avg_r >= RECALL_THRESHOLD
    rw_f1 = avg_p if meets_target else avg_r * 0.5
    return SweepResult(top_k_tables, top_k_columns, avg_r, avg_p, f1, rw_f1, meets_target, len(dev_subset))


def _print_sweep_table(results):
    W = 74
    header = f"{'tables':>8} {'cols':>6} {'recall':>9} {'precision':>11} {'F1':>8} {'meets90%':>9}  note"
    print("\n" + "=" * W)
    print("SWEEP RESULTS — ranked by recall-weighted score")
    print("=" * W)
    print(header); print("-" * W)
    for i, r in enumerate(sorted(results, key=lambda r: r.recall_weighted_f1, reverse=True)):
        target_str = "YES" if r.meets_recall_target else "NO "
        note = "<-- RECOMMENDED" if i == 0 else (f"recall too low (<{RECALL_THRESHOLD*100:.0f}%)" if not r.meets_recall_target else "")
        print(f"{r.top_k_tables:>8} {r.top_k_columns:>6} {r.avg_recall*100:>8.2f}% "
              f"{r.avg_precision*100:>10.2f}% {r.f1*100:>7.2f}% {target_str:>9}  {note}")
    print("=" * W)


def run_sweep_and_get_best(sample_ratio, cfg):
    logger.info("Loading schema for sweep …")
    import pandas as pd
    schema_df = load_spider_schema(cfg.tables_json)
    graph = build_schema_graph(schema_df)
    with open(cfg.dev_json, "r", encoding="utf-8") as f:
        dev_data = json.load(f)
    n = max(1, int(len(dev_data) * sample_ratio))
    rng = np.random.default_rng(cfg.seed)
    indices = rng.choice(len(dev_data), size=n, replace=False)
    dev_subset = [dev_data[i] for i in sorted(indices)]
    logger.info("Sweep subset: %d / %d questions", n, len(dev_data))
    logger.info("Loading embedding model for sweep: %s", cfg.embedding_model)
    embed_model = SentenceTransformer(cfg.embedding_model)
    unique_db_ids = list(dict.fromkeys(item["db_id"] for item in dev_subset))
    schema_cache = {db_id: build_schema_index(graph, db_id, embed_model)
                    for db_id in tqdm(unique_db_ids, desc="Sweep — indexing")}
    combos = list(product(TOP_K_TABLES_VALUES, TOP_K_COLUMNS_VALUES))
    results = []
    for top_k_tables, top_k_columns in tqdm(combos, desc="Sweep — grid"):
        result = _run_sweep_combination(top_k_tables, top_k_columns, dev_subset, graph, schema_cache, embed_model, cfg)
        results.append(result)
        logger.info("  tables=%d cols=%d → recall=%.2f%% precision=%.2f%% F1=%.2f%%",
                    top_k_tables, top_k_columns, result.avg_recall * 100, result.avg_precision * 100, result.f1 * 100)
    _print_sweep_table(results)
    best = max(results, key=lambda r: r.recall_weighted_f1)
    logger.info("Best config: top_k_tables=%d top_k_columns=%d recall=%.2f%% precision=%.2f%%",
                best.top_k_tables, best.top_k_columns, best.avg_recall * 100, best.avg_precision * 100)
    return best.top_k_tables, best.top_k_columns

## 8. Baseline: Table-Level Graph (`baseline.py`)

Coarser retrieval — each node is a whole table instead of a single column.

In [ ]:
@dataclass
class TableSchemaIndex:
    db_name: str
    table_node_ids: list[str]
    table_embeddings: torch.Tensor


@dataclass
class BaselineResult:
    index: int
    db_id: str
    question: str
    gold_sql: str
    pred_sql: str
    recall: float
    precision: float


def build_table_graph(schema_df) -> nx.Graph:
    G = nx.Graph()
    df_clean = schema_df[schema_df["Column"] != "*"].copy()
    for (db, table), group in df_clean.groupby(["Database", "Table"]):
        node_id = f"{db}.{table}"
        columns = group[["Column", "Type", "PK"]].rename(columns={"PK": "is_pk"}).to_dict("records")
        G.add_node(node_id, database=db, table=table, columns=columns, type="table")
    fk_rows = df_clean[df_clean["FK_Relation"] != "-"]
    for _, row in fk_rows.iterrows():
        src = f"{row['Database']}.{row['Table']}"
        parts = row["FK_Relation"].split(".")
        if len(parts) != 2:
            continue
        tgt_table, tgt_col = parts
        tgt = f"{row['Database']}.{tgt_table}"
        if G.has_node(src) and G.has_node(tgt) and not G.has_edge(src, tgt):
            G.add_edge(src, tgt, relation="foreign_key", from_col=row["Column"], to_col=tgt_col)
    logger.info("Table graph built: %d table-nodes, %d FK-edges", G.number_of_nodes(), G.number_of_edges())
    return G


def build_table_index(graph, db_name, embed_model):
    nodes = [(n, d) for n, d in graph.nodes(data=True)
             if d.get("database") == db_name and d.get("type") == "table"]
    if not nodes:
        return TableSchemaIndex(db_name, [], torch.zeros(0))
    table_node_ids, texts = [], []
    for node_id, d in nodes:
        col_names = " ".join(c["Column"] for c in d["columns"])
        texts.append(f"table {d['table']} columns {col_names}")
        table_node_ids.append(node_id)
    embeddings = embed_model.encode(texts, convert_to_tensor=True)
    return TableSchemaIndex(db_name=db_name, table_node_ids=table_node_ids, table_embeddings=embeddings)


def semantic_linking_table_level(index, user_query, embed_model, top_k=3):
    if not index.table_node_ids:
        return []
    query_emb = embed_model.encode([f"query {user_query}"], convert_to_tensor=True)
    scores = util.cos_sim(query_emb, index.table_embeddings)[0]
    k = min(top_k, len(index.table_node_ids))
    return [index.table_node_ids[i] for i in scores.topk(k).indices.tolist()]


def trace_table_paths(graph, detected_table_nodes):
    all_nodes = set(detected_table_nodes)
    for i in range(len(detected_table_nodes)):
        for j in range(i + 1, len(detected_table_nodes)):
            try:
                path = nx.shortest_path(graph, detected_table_nodes[i], detected_table_nodes[j])
                all_nodes.update(path)
            except nx.NetworkXNoPath:
                pass
    return list(all_nodes)


def build_table_schema_context(graph, table_nodes):
    lines = []
    for node in table_nodes:
        if node not in graph.nodes:
            continue
        d = graph.nodes[node]
        lines.append(f"CREATE TABLE {d['table']} (")
        col_lines = []
        for col in d["columns"]:
            line = f"    {col['Column']} {col['Type']}"
            if col.get("is_pk") == "Yes":
                line += " PRIMARY KEY"
            col_lines.append(line)
        fk_lines = []
        for _, neighbor, ed in graph.edges(node, data=True):
            if ed.get("relation") == "foreign_key" and ed.get("from_col"):
                neighbor_table = graph.nodes[neighbor]["table"]
                fk_lines.append(f"    FOREIGN KEY ({ed['from_col']}) REFERENCES {neighbor_table}({ed['to_col']})")
        lines.append(",\n".join(col_lines + fk_lines))
        lines.append(");\n")
    return "\n".join(lines)


def evaluate_table_linking(gold_sql, retrieved_table_nodes, graph, db_name):
    clean_sql = re.sub(r"[^\w\s]", " ", gold_sql.lower())
    sql_tokens = set(clean_sql.split())
    gold_elements: set[str] = set()
    for n, d in graph.nodes(data=True):
        if d.get("database") != db_name:
            continue
        t_name = d["table"].lower()
        if t_name in sql_tokens:
            gold_elements.add(t_name)
        for col in d.get("columns", []):
            c_name = col["Column"].lower()
            if c_name in sql_tokens and c_name != "*":
                gold_elements.add(c_name)
    if not gold_elements:
        return 1.0, 1.0
    retrieved_elements: set[str] = set()
    for node in retrieved_table_nodes:
        if node not in graph.nodes:
            continue
        d = graph.nodes[node]
        retrieved_elements.add(d["table"].lower())
        for col in d.get("columns", []):
            retrieved_elements.add(col["Column"].lower())
    if not retrieved_elements:
        return 0.0, 0.0
    hit = gold_elements & retrieved_elements
    return len(hit) / len(gold_elements), len(hit) / len(retrieved_elements)


def run_single_baseline(question, gold_sql, db_id, graph, embed_model, model, tokenizer, cfg, table_index):
    if cfg.use_full_schema_bypass:
        table_nodes = [n for n, d in graph.nodes(data=True)
                       if d.get("database") == db_id and d.get("type") == "table"]
    else:
        detected = semantic_linking_table_level(table_index, question, embed_model,
                                                top_k=cfg.top_k_tables if cfg.top_k_tables > 0 else 3)
        if not detected:
            table_nodes = [n for n, d in graph.nodes(data=True)
                           if d.get("database") == db_id and d.get("type") == "table"]
        else:
            table_nodes = trace_table_paths(graph, detected)
    if not table_nodes:
        return "SELECT 1", 0.0, 0.0
    recall, precision = evaluate_table_linking(gold_sql, table_nodes, graph, db_id)
    schema_context = build_table_schema_context(graph, table_nodes)
    extracted_values = {"strings": re.findall(r"'([^']*)'", question), "numbers": re.findall(r"\d+", question)}
    prompt = build_prompt(question, schema_context, extracted_values, few_shot_block="")
    pred_sql = generate_sql(prompt, model, tokenizer, cfg)
    return pred_sql, recall, precision

## 9. Main Pipeline (`pipeline.py`)

In [ ]:
@dataclass
class PipelineResult:
    index: int
    db_id: str
    question: str
    gold_sql: str
    pred_sql: str
    recall: float
    precision: float
    retrieved_schema: str = ""
    gold_schema: str = ""


def _bar(value: float, width: int = 20) -> str:
    filled = round(value * width)
    return "[" + "#" * filled + "." * (width - filled) + "]"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def build_gold_schema_context(gold_sql, graph, db_id):
    gold_elements = _parse_gold_elements(gold_sql, graph, db_id)
    gold_nodes = [n for n, d in graph.nodes(data=True)
                  if d.get("database") == db_id and d.get("type") == "column"
                  and (d["table"].lower() in gold_elements or d["column"].lower() in gold_elements)]
    if not gold_nodes:
        return "(could not parse gold schema)"
    return build_schema_context(graph, gold_nodes)


def run_single(question, gold_sql, db_id, graph, embed_model, model, tokenizer, cfg,
               schema_index=None, few_shot_index=None):
    if schema_index is None:
        schema_index = build_schema_index(graph, db_id, embed_model)
    if cfg.use_full_schema_bypass:
        column_nodes = [n for n, d in graph.nodes(data=True)
                        if d.get("database") == db_id and d.get("type") == "column"]
    else:
        detected_cols = semantic_schema_linking(
            graph, db_id, question, all_columns=schema_index.columns,
            col_embeddings=schema_index.col_embeddings, embed_model=embed_model, cfg=cfg, index=schema_index)
        if not detected_cols:
            column_nodes = [n for n, d in graph.nodes(data=True)
                            if d.get("database") == db_id and d.get("type") == "column"]
        else:
            c_nodes, paths, _ = trace_schema_paths(graph, db_id, detected_cols)
            all_path_nodes = list(set(c_nodes + [node for path in paths for node in path]))
            detected_col_names = {c.lower() for c in detected_cols}
            column_nodes = prune_path_nodes(graph, detected_col_names, all_path_nodes) or all_path_nodes
    if not column_nodes:
        return "SELECT 1", 0.0, 0.0, "", []
    recall, precision = evaluate_schema_linking(gold_sql, column_nodes, graph, db_id)
    few_shot_block = ""
    if few_shot_index is not None and cfg.few_shot_k > 0:
        examples = retrieve_few_shot_examples(question, db_id, few_shot_index, embed_model, cfg)
        few_shot_block = format_few_shot_block(examples)
    schema_context = build_schema_context(graph, column_nodes)
    extracted_values = {"strings": re.findall(r"'([^']*)'", question), "numbers": re.findall(r"\d+", question)}
    prompt = build_prompt(question, schema_context, extracted_values, few_shot_block)
    pred_sql = generate_sql(prompt, model, tokenizer, cfg)
    return pred_sql, recall, precision, schema_context, column_nodes


def run_official_evaluation(cfg):
    evaluator = Path("evaluation.py")
    if not evaluator.exists():
        logger.warning("evaluation.py not found. Download with:\n"
                       "  wget https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py\n"
                       "  wget https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py")
        return
    common_args = ["--gold", str(cfg.gold_sql), "--pred", str(cfg.predictions_file),
                   "--db", str(cfg.db_dir), "--table", str(cfg.tables_json)]
    print("\n" + "=" * 60); print("OFFICIAL SPIDER EVALUATION (Exact Match)"); print("=" * 60)
    subprocess.run(["python", "evaluation.py"] + common_args + ["--etype", "match"], check=False)
    print("\n" + "=" * 60); print("OFFICIAL SPIDER EVALUATION (Execution Accuracy)"); print("=" * 60)
    subprocess.run(["python", "evaluation.py"] + common_args + ["--etype", "exec"], check=False)


def print_schema_linking_summary(results):
    valid = [r for r in results if r.recall >= 0]
    if not valid: return
    avg_recall = sum(r.recall for r in valid) / len(valid) * 100
    avg_precision = sum(r.precision for r in valid) / len(valid) * 100
    print("\n" + "=" * 60); print("SCHEMA LINKING SUMMARY (GraphRAG)"); print("=" * 60)
    print(f"  Recall    : {avg_recall:.2f}%"); print(f"  Precision : {avg_precision:.2f}%")
    print(f"  Samples   : {len(valid)}")

## 10. Run Pipeline

Configurable cell — adjust flags then run.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
USE_FULL_SCHEMA = False   # True = bypass GraphRAG (ablation)
SKIP_SWEEP      = False   # True = skip precision sweep, use config values
SWEEP_SAMPLE    = 0.2     # fraction of dev set used for sweep
RUN_BASELINE    = False   # True = also run table-level baseline for comparison
SAMPLE_RATIO    = 1.0     # fraction of dev set to evaluate (1.0 = full)

import pandas as pd
cfg = PipelineConfig(use_full_schema_bypass=USE_FULL_SCHEMA)
set_seed(cfg.seed)

# ── Step 0: Precision sweep ─────────────────────────────────────────────────
if not SKIP_SWEEP and not USE_FULL_SCHEMA:
    logger.info("=" * 60)
    logger.info("STEP 0: Precision sweep (pass SKIP_SWEEP=True to skip)")
    logger.info("=" * 60)
    best_tables, best_cols = run_sweep_and_get_best(sample_ratio=SWEEP_SAMPLE, cfg=cfg)
    logger.info("Sweep done — updating config: top_k_tables=%d, top_k_columns=%d", best_tables, best_cols)
    cfg.top_k_tables  = best_tables
    cfg.top_k_columns = best_cols
else:
    logger.info("Skipping sweep — using: top_k_tables=%d, top_k_columns=%d",
                cfg.top_k_tables, cfg.top_k_columns)

# ── Step 1: Load schema & graph ─────────────────────────────────────────────
logger.info("Loading schema and building graph …")
schema_df = load_spider_schema(cfg.tables_json)
graph = build_schema_graph(schema_df)

# ── Step 2: Load dev set ────────────────────────────────────────────────────
logger.info("Loading Spider dev set …")
with open(cfg.dev_json, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

if SAMPLE_RATIO < 1.0:
    n = max(1, int(len(dev_data) * SAMPLE_RATIO))
    rng = np.random.default_rng(cfg.seed)
    dev_data = [dev_data[i] for i in sorted(rng.choice(len(dev_data), size=n, replace=False))]
logger.info("Evaluating %d questions", len(dev_data))

# ── Step 3: Load models ─────────────────────────────────────────────────────
logger.info("Loading embedding model: %s", cfg.embedding_model)
embed_model = SentenceTransformer(cfg.embedding_model)

logger.info("Loading LLM: %s (4-bit quantised)", cfg.llm_model)
llm, tokenizer = load_model_and_tokenizer(cfg)

# ── Step 4: Pre-build schema indices ────────────────────────────────────────
unique_db_ids = list(dict.fromkeys(item["db_id"] for item in dev_data))
logger.info("Pre-building schema indices for %d databases …", len(unique_db_ids))
schema_cache = {db_id: build_schema_index(graph, db_id, embed_model)
                for db_id in tqdm(unique_db_ids, desc="Indexing schemas")}

# ── Step 5: Build few-shot index ────────────────────────────────────────────
few_shot_idx = None
if cfg.few_shot_k > 0:
    logger.info("Building few-shot index from training set …")
    few_shot_idx = build_few_shot_index(cfg.train_json, embed_model)
else:
    logger.info("Few-shot disabled (few_shot_k=0), running zero-shot.")

# ── Step 6: Main loop ───────────────────────────────────────────────────────
results: list[PipelineResult] = []
W = 72

with open(cfg.predictions_file, "w", encoding="utf-8") as pred_file:
    for i, item in enumerate(tqdm(dev_data, desc="Generating SQL")):
        db_id, question, gold_sql = item["db_id"], item["question"], item["query"]
        try:
            pred_sql, recall, precision, retrieved_schema, column_nodes = run_single(
                question, gold_sql, db_id, graph, embed_model, llm, tokenizer, cfg,
                schema_index=schema_cache[db_id], few_shot_index=few_shot_idx)
            gold_schema = build_gold_schema_context(gold_sql, graph, db_id)
        except Exception:
            logger.exception("Error on sample %d (db=%s)", i, db_id)
            pred_sql, recall, precision, retrieved_schema, gold_schema = "SELECT 1", 0.0, 0.0, "", ""

        results.append(PipelineResult(i+1, db_id, question, gold_sql, pred_sql, recall, precision,
                                       retrieved_schema, gold_schema))
        pred_file.write(f"{pred_sql}\n")

        em_mark = "MATCH" if gold_sql.strip().lower() == pred_sql.strip().lower() else "DIFF"
        print(f"\n{'='*W}")
        print(f"  [{i+1:>4}/{len(dev_data)}]  DB: {db_id:<24} {em_mark}")
        print(f"{'='*W}")
        print(f"  Q        : {question}")
        print(f"{'-'*W}")
        print(f"  Recall   : {_bar(recall)}  {recall*100:5.1f}%")
        print(f"  Precision: {_bar(precision)}  {precision*100:5.1f}%")
        print(f"{'-'*W}")
        print(f"  GOLD SQL : {gold_sql}")
        print(f"  PRED SQL : {pred_sql}")
        print(f"{'='*W}")

logger.info("Predictions saved to %s", cfg.predictions_file)

# ── Step 7: Summary & evaluation ────────────────────────────────────────────
print_schema_linking_summary(results)
run_official_evaluation(cfg)

## 11. (Optional) Few-Shot Ablation (`ablation.py`)

Compare k=0 vs k=1 vs k=3 vs k=5 few-shot examples.

In [ ]:
# Set K_VALUES and ABLATION_SAMPLE to run
K_VALUES        = [0, 1, 3, 5]
ABLATION_SAMPLE = 0.2   # fraction of dev set

# ── Setup (reuses variables from Cell 10 if already run) ───────────────────
try:
    graph, embed_model, llm, tokenizer, schema_cache
    logger.info("Reusing models from main pipeline run.")
except NameError:
    import pandas as pd
    cfg = PipelineConfig()
    schema_df = load_spider_schema(cfg.tables_json)
    graph = build_schema_graph(schema_df)
    embed_model = SentenceTransformer(cfg.embedding_model)
    llm, tokenizer = load_model_and_tokenizer(cfg)

with open(cfg.dev_json, "r", encoding="utf-8") as f:
    dev_data_full = json.load(f)

n = max(1, int(len(dev_data_full) * ABLATION_SAMPLE))
rng_ab = np.random.default_rng(42)
ablation_subset = [dev_data_full[i] for i in sorted(rng_ab.choice(len(dev_data_full), size=n, replace=False))]
logger.info("Ablation subset: %d questions", len(ablation_subset))

unique_ab_dbs = list(dict.fromkeys(item["db_id"] for item in ablation_subset))
schema_cache_ab = {db_id: build_schema_index(graph, db_id, embed_model)
                   for db_id in tqdm(unique_ab_dbs, desc="Indexing")}

few_shot_index_ab = None
if any(k > 0 for k in K_VALUES):
    logger.info("Building few-shot index …")
    few_shot_index_ab = build_few_shot_index(cfg.train_json, embed_model)

@dataclass
class AblationResult:
    k: int
    avg_recall: float
    avg_precision: float
    n_samples: int
    predictions_file: Path

ablation_results = []
for k in K_VALUES:
    logger.info("Running ablation k=%d …", k)
    ab_cfg = PipelineConfig(data_path=cfg.data_path, top_k_tables=cfg.top_k_tables,
                             top_k_columns=cfg.top_k_columns, few_shot_k=k,
                             few_shot_same_db_first=cfg.few_shot_same_db_first)
    pred_path = Path(f"ablation_predictions_k{k}.txt")
    recalls, precisions = [], []
    with open(pred_path, "w", encoding="utf-8") as pf:
        for item in tqdm(ablation_subset, desc=f"k={k}", leave=False):
            db_id, question, gold_sql = item["db_id"], item["question"], item["query"]
            try:
                pred_sql, recall, precision, _, _ = run_single(
                    question, gold_sql, db_id, graph, embed_model, llm, tokenizer, ab_cfg,
                    schema_index=schema_cache_ab[db_id],
                    few_shot_index=few_shot_index_ab if k > 0 else None)
                recalls.append(recall); precisions.append(precision)
            except Exception:
                logger.exception("Error on ablation sample"); pred_sql = "SELECT 1"
                recalls.append(0.0); precisions.append(0.0)
            pf.write(f"{pred_sql}\n")
    ablation_results.append(AblationResult(k, float(np.mean(recalls)), float(np.mean(precisions)),
                                            len(ablation_subset), pred_path))
    logger.info("k=%d → recall=%.2f%% precision=%.2f%%", k, ablation_results[-1].avg_recall*100,
                ablation_results[-1].avg_precision*100)

print("\n" + "=" * 60); print("FEW-SHOT ABLATION RESULTS"); print("=" * 60)
print(f"{'k':>5} {'recall':>9} {'precision':>11}  file")
print("-" * 60)
for r in sorted(ablation_results, key=lambda x: x.k):
    print(f"{r.k:>5} {r.avg_recall*100:>8.2f}% {r.avg_precision*100:>10.2f}%  {r.predictions_file.name}")
print("=" * 60)

## 12. (Optional) GraphRAG vs Baseline Comparison

Run both pipelines side-by-side on the same sample.

In [ ]:
COMPARE_SAMPLE = 0.2   # fraction of dev set

try:
    graph, embed_model, llm, tokenizer
    logger.info("Reusing models.")
except NameError:
    import pandas as pd
    cfg = PipelineConfig()
    schema_df = load_spider_schema(cfg.tables_json)
    graph = build_schema_graph(schema_df)
    embed_model = SentenceTransformer(cfg.embedding_model)
    llm, tokenizer = load_model_and_tokenizer(cfg)

with open(cfg.dev_json, "r", encoding="utf-8") as f:
    dev_data_full = json.load(f)

n = max(1, int(len(dev_data_full) * COMPARE_SAMPLE))
rng_cmp = np.random.default_rng(cfg.seed)
cmp_subset = [dev_data_full[i] for i in sorted(rng_cmp.choice(len(dev_data_full), size=n, replace=False))]
logger.info("Comparison subset: %d questions", len(cmp_subset))

unique_cmp_dbs = list(dict.fromkeys(item["db_id"] for item in cmp_subset))
col_cache = {db_id: build_schema_index(graph, db_id, embed_model)
             for db_id in tqdm(unique_cmp_dbs, desc="Indexing columns")}

tbl_graph = build_table_graph(schema_df)
tbl_cache = {db_id: build_table_index(tbl_graph, db_id, embed_model)
             for db_id in tqdm(unique_cmp_dbs, desc="Indexing tables")}

g_results, b_results = [], []
with open("predictions.txt", "w") as gf, open("baseline_predictions.txt", "w") as bf:
    for i, item in enumerate(tqdm(cmp_subset, desc="Comparing")):
        db_id, question, gold_sql = item["db_id"], item["question"], item["query"]
        try:
            g_pred, g_r, g_p, _, _ = run_single(question, gold_sql, db_id, graph, embed_model,
                                                  llm, tokenizer, cfg, schema_index=col_cache[db_id])
        except Exception:
            g_pred, g_r, g_p = "SELECT 1", 0.0, 0.0
        try:
            b_pred, b_r, b_p = run_single_baseline(question, gold_sql, db_id, tbl_graph, embed_model,
                                                     llm, tokenizer, cfg, table_index=tbl_cache[db_id])
        except Exception:
            b_pred, b_r, b_p = "SELECT 1", 0.0, 0.0
        g_results.append(PipelineResult(i+1, db_id, question, gold_sql, g_pred, g_r, g_p))
        b_results.append(BaselineResult(i+1, db_id, question, gold_sql, b_pred, b_r, b_p))
        gf.write(g_pred.strip() + "\n"); bf.write(b_pred.strip() + "\n")

g_r_avg  = np.mean([r.recall    for r in g_results]) * 100
g_p_avg  = np.mean([r.precision for r in g_results]) * 100
b_r_avg  = np.mean([r.recall    for r in b_results]) * 100
b_p_avg  = np.mean([r.precision for r in b_results]) * 100

print("\n" + "=" * 70)
print("  COMPARISON REPORT — GraphRAG (column/node) vs Baseline (table/node)")
print("=" * 70)
print(f"  {'Metric':<22} {'GraphRAG':>12}  {'Baseline':>12}  {'Delta (G-B)':>12}")
print("-" * 70)
print(f"  {'Schema Recall':<22} {g_r_avg:>11.2f}%  {b_r_avg:>11.2f}%  {g_r_avg-b_r_avg:>+11.2f}%")
print(f"  {'Schema Precision':<22} {g_p_avg:>11.2f}%  {b_p_avg:>11.2f}%  {g_p_avg-b_p_avg:>+11.2f}%")
print("-" * 70)
print("  Node granularity      column/node      table/node")
print("  Linking strategy      n-gram match     single query embed")
print("=" * 70)